Learning goals:
1. Set up a coupled flow and transport 2d simulation.
2. Import of fractures from a file
3. Mesh construction
4. Run simulation
5. 

# Coupled flow and transport in a fractured porous medium
Here we show how to set up a simulation of coupled flow and transport in a fractured porous media using the multiphysics simulation framework in PorePy. The tutorial covers how to specify:
* The geometry of the fracture network and the computational domain;
* Parameters for mesh size control;
* Parameters for permeability; porosity etc.;
* Boundary conditions;
* Simulation time etc.

Although the tutorial in one sense aims to be self-contained, we will make frequent references to other tutorials in PorePy proper, where more details are provided. For instance, it may be advisable to read through the PorePy tutorial on single phase flow before reading further here.

A word of caution: The multiphysics models in PorePy make heavy use of Python mixins (for readers familiar with object oriented programming, mixins can crudely be thought of as inheritance). While this is a powerful technology that allows for code reuse and flexibility in simulation setup, it will lead to unexpected behavior (typically the calling of the wrong version of a polymorphic method). The below code is safe to use, but before composing simulation classes from scratch, it is highly advisable to read up on mixins - the tutorial on single phase flow contains an excellent reference in that regard.

# Specifying a simulation
We start by importing PorePy, as well as numpy and the Path class from pathlib

In [1]:
import porepy as pp
import numpy as np
from pathlib import Path

## Geometry
We will specify the simulation setup through a series of mixin classes. First, we import the fracture network geometry from a csv file and also define the domain.

In [ ]:
class Geometry:

    def set_fractures(self):
        # The first line in the csv file specifies the domain, so we skip it when
        # reading the fracture data.
        data = np.genfromtxt(Path('fractures.csv'), delimiter=',', skip_header=1)
        fractures = []
        for row in data:
            f = pp.LineFracture(data.reshape((2, -1), order="F"))
                
            fractures.append(f)
        self._fractures = fractures

    def set_domain(self):
        # Read only the first line of the csv file.
        data = np.genfromtxt(Path('fractures.csv'), delimiter=',', max_rows=1)
        self._domain = pp.Domain({"xmin": data[0], "ymin": data[1], "xmax": data[2], "ymax": data[3]})

## Boundary conditions
By default, PorePy assign no-flow (homogeneous Neumann) conditions for flow and transport. We therefore only need to specify the conditions along boundaries where other conditions apply. In our case, we will (arbitrarily) set the pressure at the left and right boundaries to 2MPa and 1MPa, respectively, and leave the top and bottom boundaries untouched.